# RNA Modification Pipeline

## Description
Extract A-to-I induced modifications for two samples from Oxford Nanopore .bam file readings. Was able to successfully extract ~9500 genes of interest with modification. [View here](https://rna-analysis.vercel.app/)

## Samples
- MR01-1: T-Cell Leukemia Samples
- MR01-2: iPS Generated T-Cell Samples

## Modification Definitions
- A+17596: Adenosine to Inosine modification
- A+a: Adenosine methylation (m6a)

## Tech Stack
(Processing):
- Python
- modkit
- samtools
  
(Webpage):
- React
- Supabase
- Vercel

## References
- https://nanoporetech.github.io/modkit/quick_start.html
- https://samtools.github.io/hts-specs/SAMv1.pdf
- https://genome.ucsc.edu/ 

## Processing

### Bam file Verification

Verify .bam files are reading correctly.

In [ ]:
modkit summary MR01-1.dorado.aligned.sorted.bam

> sampling 10042 reads from BAM
> calculating threshold at 10(th) percentile
> calculated thresholds: A: 0.55078125
# bases             A 
# total_reads_used  10042 
# count_reads_A     10042 
# pass_threshold_A  0.55078125 
 base  code   pass_count  pass_frac    all_count  all_frac 
 A     -      1162975     0.8937457    1252363    0.8665025 
 A     a      112306      0.08630711   150224     0.10393909 
 A     17596  25956       0.019947173  42721      0.029558405 

Make index files for the MR01 datasets

In [ ]:
samtools index -M *.bam

Read head of sample to verify readings.

In [ ]:
import pysam
import pandas as pd

bamfile = pysam.AlignmentFile("./Data/MR01-1.dorado.aligned.sorted.bam", "rb")

records = []

max_reads = 10

for i, read in enumerate(bamfile):
    if i > max_reads:
        break
        
    data = {
        "query_name": read.query_name,
        "reference_name": bamfile.get_reference_name(read.reference_id) if not read.is_unmapped else None,
        "reference_start": read.reference_start,
        "mapping_quality": read.mapping_quality,
        "cigarstring": read.cigarstring,
        "query_sequence": read.query_sequence,
        "flag": read.flag,
        "is_unmapped": read.is_unmapped,
        "is_reverse": read.is_reverse,
        "is_read1": read.is_read1,
        "is_read2": read.is_read2,
        "template_length": read.template_length,
        "query_qualities": read.query_qualities,
        "next_reference_name": bamfile.get_reference_name(read.next_reference_id) if read.next_reference_id != -1 else None,
        "next_reference_start": read.next_reference_start
    }

    try:
        tags = dict(read.tags)
        data.update(tags)
    except AttributeError:
        pass  # No tags found

    records.append(data)

bamfile.close()

df = pd.DataFrame(records)

pd.set_option('display.max_columns', None)
print(df.head())

In [ ]:
                             query_name reference_name  reference_start  \
0  2999fe64-95e8-4fd6-aa38-21bdc7301fbe           chr1            12008   
1  e8dc9394-2557-4a77-99ba-faab86b3ea96           chr1            12594   
2  d4d46458-b3b6-43bf-966c-7b768c49096b           chr1            13169   
3  2999fe64-95e8-4fd6-aa38-21bdc7301fbe           chr1            13465   
4  fb350f7d-6765-460b-92d8-3d066ecdbfa5           chr1            13473   

   mapping_quality                                        cigarstring  \
0                1                     18H6M2I71M2D63M1D54M3D20M1081H   
1                1                           25H9M1D33M1D14M2D68M920H   
2                0  5S53M1D40M6D158M3D81M3D26M1I207M4I64M2D2M1D34M...   
3                9  357S48M1I72M1I55M2D40M1D49M1I2M2I4M1D8M1I2M1I1...   
4               27  8S36M1I9M2D11M1D150M1D20M1D43M2I1M2I2M2I65M1D1...   

                                      query_sequence  flag  is_unmapped  \
0  GGTGTCCTTGACTTCCAGCAACTGCTGGCCTGTGCCAGGGTGGAAG...  2048        False   
1  GCTCCTGTCCACCCCAGGTGTGTGGTGATGCCAGGCATGCCCTCTC...  2048        False   
2                                               None   256        False   
3  AACGAGATTGCCAGCCACGGTGTCCTTGACTTCCAGCAACTGCTGG...     0        False   
4  TTCCTTTCTCCTGACAGGCAGCTGCACCACTGCCTGGCGCTGCGCC...     0        False   

   is_reverse  is_read1  is_read2  template_length  \
0       False     False     False                0   
1       False     False     False                0   
2       False     False     False                0   
3       False     False     False                0   
4       False     False     False                0   

                                     query_qualities next_reference_name  \
0  [17, 16, 13, 11, 8, 11, 12, 13, 31, 33, 31, 30...                None   
1  [27, 27, 27, 32, 31, 29, 24, 15, 14, 15, 18, 3...                None   
2                                               None                None   
3  [12, 17, 18, 20, 21, 21, 21, 21, 21, 17, 10, 8...                None   
4  [3, 6, 6, 7, 11, 12, 11, 11, 18, 21, 21, 22, 2...                None   

   next_reference_start         qs        du     ns    ts  mx    ch  \
0                    -1  15.599547   9.43625  37745  1900   3  2387   
1                    -1  14.338394   8.52450  34098  2050   4  1331   
2                    -1  16.614334  12.22900  48916  2100   4   360   
3                    -1  15.599547   9.43625  37745  1900   3  2387   
4                    -1  15.124628  11.79625  47185  1900   2   631   

                              st     rn             fn          sm  \
0  2024-04-27T05:46:09.362+00:00  16830  PAW01364.pod5  781.444641   
1  2024-04-27T20:32:04.579+00:00  61633  PAW01364.pod5  807.444641   
2  2024-04-27T13:46:56.681+00:00  36344  PAW01364.pod5  820.444641   
3  2024-04-27T05:46:09.362+00:00  16830  PAW01364.pod5  781.444641   
4  2024-04-27T12:28:41.066+00:00  36904  PAW01364.pod5  819.444641   

           sd  sv  dx                                                 RG  \
0  116.090813  pa   0  64de1be79fe30ea5bf08647394f04c02b19eb5f2_rna00...   
1  116.090813  pa   0  64de1be79fe30ea5bf08647394f04c02b19eb5f2_rna00...   
2  116.090813  pa   0  64de1be79fe30ea5bf08647394f04c02b19eb5f2_rna00...   
3  116.090813  pa   0  64de1be79fe30ea5bf08647394f04c02b19eb5f2_rna00...   
4  116.090813  pa   0  64de1be79fe30ea5bf08647394f04c02b19eb5f2_rna00...   

                                                  mv  NM    ms    AS  nn  \
0  [5, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...  16   352   348   0   
1  [5, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...   6   217   216   0   
2  [5, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...  70  1057  2102   0   
3  [5, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...  47  1616  1612   0   
4  [5, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, ...  41   841  1638   0   

         de tp  cm   s1     s2  \
0  0.055046  P  11  110  110.0   
1  0.039370  P   4   45   45.0   
2  0.039548  S  60  740    NaN   
3  0.043887  P  42  524  500.0   
4  0.038095  P  58  631  613.0   

                                                  MD  rl  \
0               40C36^GA33C27T1^A7A9A20A10C4^ACT5C14   0   
1                                 9^T1C31^T2C11^TC68   0   
2  53^A40^CCTGGA32C24G33G38T23A3^TTC77T3^CTT3T94T...  19   
3  42T5T71A26T27^GC40^C55^C1T9G18C45^T21C69T3G5^C...   0   
4  34T4T3T1^TG6C1G2^G0G79T69^C0C19^C34C0T31C40T2^...   0   

                                  SA      MN  \
0    chr1,13466,+,357S948M3I7S,9,47;     NaN   
1    chr1,13481,+,146S922M7D1S,1,62;     NaN   
2                                NaN     NaN   
3  chr1,12009,+,18S216M4D1081S,1,16;  1315.0   
4                                NaN   949.0   

                                                  MM  \
0                                                NaN   
1                                                NaN   
2                                                NaN   
3  A+17596.,0,0,2,0,1,0,1,0,0,1,1,0,0,0,0,1,0,0,1...   
4  A+17596.,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0...   

                                                  ML   pi  sp  
0                                                NaN  NaN NaN  
1                                                NaN  NaN NaN  
2                                                NaN  NaN NaN  
3  [4, 18, 15, 2, 0, 0, 0, 64, 55, 1, 0, 0, 1, 2,...  NaN NaN  
4  [58, 99, 19, 23, 90, 53, 90, 43, 44, 23, 34, 1...  NaN NaN 

*Focusing mostly on MM (modification locations on reading) and ML (modification probabilities on reading).

### Pickling Tabulated Chromosomes

We want to analyze and tabulate our chromosomes and readings in Pandas. We can't do this in raw .bam files, so, we will pickle them and then read them into pandas.

Done for both MR01-1 and MR01-2. Processing for MR01-1 shown below.

Read in chr4 as test.

In [ ]:
import pysam
import pandas as pd

# Open BAM file
bamfile = pysam.AlignmentFile("../MR01-1.dorado.aligned.sorted.bam", "rb")

records = []
# max_reads = 10 

try:
    for i, read in enumerate(bamfile.fetch("chr4")):
        # if i >= max_reads:
        #     break

        data = {
            "query_name": read.query_name,
            "reference_name": bamfile.get_reference_name(read.reference_id) if not read.is_unmapped else None,
            "reference_start": read.reference_start,
            "mapping_quality": read.mapping_quality,
            "cigarstring": read.cigarstring,
            "query_sequence": read.query_sequence,
            "flag": read.flag,
            "is_unmapped": read.is_unmapped,
            "is_reverse": read.is_reverse,
            "is_read1": read.is_read1,
            "is_read2": read.is_read2,
            "template_length": read.template_length,
            "query_qualities": read.query_qualities,
            "next_reference_name": bamfile.get_reference_name(read.next_reference_id) if read.next_reference_id != -1 else None,
            "next_reference_start": read.next_reference_start
        }

        try:
            tags = dict(read.tags)
            data.update(tags)
        except AttributeError:
            pass  # No tags

        records.append(data)
except ValueError as e:
    print(f"Error: {e}\nMake sure 'chr4' exists in the BAM header.")

bamfile.close()

df = pd.DataFrame(records)

pd.set_option('display.max_columns', None)
print(df.head())

In [ ]:
  query_name reference_name  reference_start  \
0  dc4565d1-0562-4617-b8fc-07f8306de0ca           chr4            17223   
1  4a6cada8-005d-4554-a2af-362d8dc887e5           chr4            19034   
2  754681e6-7b27-4cb1-a01d-c3246bab4d67           chr4            19117   
3  e74b3cf5-22e2-436e-a6b8-733c4d6606ad           chr4            19374   
4  4f95204a-c400-4ab3-8adb-c251071f4c78           chr4            19519   

   mapping_quality                                        cigarstring  \
0                0                  17S30M1D30M2D29M2D28M1I8M1I105M3S   
1                0  74M1D71M1D58M1D86M2D95M1D65M2D59M1D2M3I19M2D5M...   
2                0                             3S224M1D53M2I6M1D39M3S   
3                0                           3S75M1I97M1I47M3D72M2D9M   
4                0           6S80M1D47M1D2M1D3M7D30M2I6M2I60M2I102M9S   

                                      query_sequence  flag  is_unmapped  \
0                                               None   272        False   
1                                               None   256        False   
2  CTTTTTGCTGAGATGCTGTTAATTTGTAACTTTGCCCCAGCCACTT...    16        False   
3                                               None   272        False   
4                                               None   256        False   

   is_reverse  is_read1  is_read2  template_length  \
0        True     False     False                0   
1       False     False     False                0   
2        True     False     False                0   
3        True     False     False                0   
4       False     False     False                0   

                                     query_qualities next_reference_name  \
0                                               None                None   
1                                               None                None   
2  [5, 13, 19, 23, 24, 24, 23, 20, 21, 24, 18, 17...                None   
3                                               None                None   
4                                               None                None   

   next_reference_start         qs       du     ns    ts  mx    ch  \
0                    -1  17.880787  2.05225   8209  2550   4  1865   
1                    -1  15.250147  7.60750  30430  2000   4  2262   
2                    -1  15.288992  2.74750  10990  2250   2   381   
3                    -1  17.640259  2.59350  10374  2150   2  2395   
4                    -1  15.188850  2.80975  11239  2550   3  2706   

                              st     rn             fn          sm  \
0  2024-04-27T03:30:05.340+00:00  12868  PAW01364.pod5  809.444641   
1  2024-04-26T23:45:30.704+00:00   5647  PAW01364.pod5  800.444641   
2  2024-04-27T02:56:42.206+00:00  11308  PAW01364.pod5  805.444641   
3  2024-04-26T22:45:37.390+00:00   6511  PAW01364.pod5  773.444641   
4  2024-04-27T23:00:44.585+00:00  69354  PAW01364.pod5  791.444641   

           sd  sv  dx                                                 RG  \
0  116.090813  pa   0  64de1be79fe30ea5bf08647394f04c02b19eb5f2_rna00...   
1  116.090813  pa   0  64de1be79fe30ea5bf08647394f04c02b19eb5f2_rna00...   
2  116.090813  pa   0  64de1be79fe30ea5bf08647394f04c02b19eb5f2_rna00...   
3  116.090813  pa   0  64de1be79fe30ea5bf08647394f04c02b19eb5f2_rna00...   
4  116.090813  pa   0  64de1be79fe30ea5bf08647394f04c02b19eb5f2_rna00...   

                                                  mv  NM   ms    AS  nn  \
0  [5, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, ...   9  416   414   0   
1  [5, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...  53  568  1666   0   
2  [5, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...   6  613   612   0   
3  [5, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, ...  16  519   516   0   
4  [5, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...  25  556   546   0   

         de tp  cm   s1                                                 MD  \
0  0.029787  S   3   56                            13A16^A30^AA29^TA24G116   
1  0.044266  S  41  499  74^A71^A58^T86^AA43A48A2^C7T23A31G1^TC0C58^T2C...   
2  0.015385  P  21  234                                    224^C59^T1G32C4   
3  0.042763  S  13  160                3G3C68T4A31G1T31A24G13A32^AGG72^AA9   
4  0.047478  S  20  221       80^A3C0C42^T2^T3^TCTTTAC30T12T0G58C0A6T40T45   

    rl     MN                                                 MM  \
0   95    NaN                                                NaN   
1  124    NaN                                                NaN   
2   48  330.0  A+17596.,0,0,0,1,0,0,0,4,1,5,0,0,0,0,0,6,0,2,1...   
3   38    NaN                                                NaN   
4   21    NaN                                                NaN   

                                                  ML     s2   SA   pi  sp  zd  
0                                                NaN    NaN  NaN  NaN NaN NaN  
1                                                NaN    NaN  NaN  NaN NaN NaN  
2  [3, 0, 0, 0, 0, 0, 1, 0, 92, 0, 0, 7, 3, 12, 1...  234.0  NaN  NaN NaN NaN  
3                                                NaN    NaN  NaN  NaN NaN NaN  
4                                                NaN    NaN  NaN  NaN NaN NaN 

Test seems valid, pickle chromosomes so we can analyze them in pandas for later.

Process the DFs for the CHR* values

In [ ]:
import pysam
import pandas as pd
import os

bamfile = pysam.AlignmentFile("../MR01-1.dorado.aligned.sorted.bam", "rb")

output_dir = "MR01-1-tab-chr_pickles"
os.makedirs(output_dir, exist_ok=True)

for chrom in bamfile.references:
    print(f"Processing {chrom}...")
    records = []

    try:
        for read in bamfile.fetch(chrom):
            data = {
                "query_name": read.query_name,
                "reference_name": bamfile.get_reference_name(read.reference_id) if not read.is_unmapped else None,
                "reference_start": read.reference_start,
                "mapping_quality": read.mapping_quality,
                "cigarstring": read.cigarstring,
                "query_sequence": read.query_sequence,
                "flag": read.flag,
                "is_unmapped": read.is_unmapped,
                "is_reverse": read.is_reverse,
                "is_read1": read.is_read1,
                "is_read2": read.is_read2,
                "template_length": read.template_length,
                "query_qualities": read.query_qualities,
                "next_reference_name": bamfile.get_reference_name(read.next_reference_id) if read.next_reference_id != -1 else None,
                "next_reference_start": read.next_reference_start
            }

            try:
                tags = dict(read.tags)
                data.update(tags)
            except AttributeError:
                pass  # No tags

            records.append(data)

        df = pd.DataFrame(records)
        outfile = os.path.join(output_dir, f"MR01-1-tab-{chrom}.pkl")
        df.to_pickle(outfile)
        print(f"Saved {chrom} with {len(df)} reads to {outfile}")

    except ValueError as e:
        print(f"Error fetching {chrom}: {e}")

bamfile.close()

Verify and read test pkl files. Chr4 shown below as example.

In [ ]:
import pickle

with open('./MR01-1-tab-chr_pickles/MR01-1-tab-chr4.pkl', 'rb') as file:
    chr4_MR01_1 = pickle.load(file)

chr4_MR01_1

In [ ]:
chr4_MR01_1[['query_name', 'query_sequence', 'MM', 'ML']]

In [ ]:
 	query_name 	query_sequence 	MM 	ML
0 	dc4565d1-0562-4617-b8fc-07f8306de0ca 	None 	NaN 	NaN
1 	4a6cada8-005d-4554-a2af-362d8dc887e5 	None 	NaN 	NaN
2 	754681e6-7b27-4cb1-a01d-c3246bab4d67 	CTTTTTGCTGAGATGCTGTTAATTTGTAACTTTGCCCCAGCCACTT... 	A+17596.,0,0,0,1,0,0,0,4,1,5,0,0,0,0,0,6,0,2,1... 	[3, 0, 0, 0, 0, 0, 1, 0, 92, 0, 0, 7, 3, 12, 1...
3 	e74b3cf5-22e2-436e-a6b8-733c4d6606ad 	None 	NaN 	NaN
4 	4f95204a-c400-4ab3-8adb-c251071f4c78 	None 	NaN 	NaN
... 	... 	... 	... 	...
372838 	5c35f926-9964-4c21-b502-8240f24997b2 	None 	NaN 	NaN
372839 	66696a1e-e30d-4663-8a42-a4246a89803f 	None 	NaN 	NaN
372840 	9aba77dd-f64f-4082-a144-81f01f624557 	None 	NaN 	NaN
372841 	c5cc796d-7e0a-433d-8541-e288935c8946 	None 	NaN 	NaN
372842 	bee4f362-2c5e-4b41-91a8-1f765d8f8723 	None 	NaN 	NaN

MM column shows the modification locations on the associated strand. For example on query 754681e6-7b27-4cb1-a01d-c3246bab4d67, the MM value is A+17596.,0,0,0,1,0,0,0,4,1,5,0,0,0,0,0,6,0,2,1. Starting from left to right, we see that the canonical base that we are looking at is Adenosine. Followed by this is a +, meaning that we are viewing the top strand. The 17596 code is a CHebi code for Inosine, which is the modified base, which results in modification sites for A-to-I. The . symbolizes sites where the base isn't modified, or has a low probability. Finally, the last string of numbers represents the modification site in index-0 ordering. Using the same example, we have [0,0,0,1,0,0,0,4], which translates to modified bases in [0,1,2,4,5,6,7,11]. As for the A+a. readings, this would specify the canonical base A readings to the m6a readings. 

The optional ML tag lists the probability of each modification listed in the MM tag being correct, in the order that they occur. The continuous probability range 0.0 to 1.0 is remapped in equal sized portions to the discrete integers 0 to 255 inclusively. Thus the probability range corresponding to integer value N is N/256 to (N + 1)/256. Each of these probabilities would be according to the listed MM tag, where the remainder would be the unmodified codes. 

> This process is also done with MR01-2.

### Extract per-position modification probabilities for each gene

We want to find the locations for each of the modifications based on the pkl files. In order to do this, we must convert the MM and ML into values that are interpretable in table format. Moreover, this would allow us to map these positions to possible Gene locations or locations of interest (exonic, intronic, utr regions, etc). 

To do this, we must first extract the max modfication for each of the positions with each read. Since each read has a MM (position column) and ML (probability column), we must calculate the most probable modification for each MM value in the read. Since there are three possible modifcations (none, m6a, or A-to-I), we take the max probability of the three and assign that to be the most likely modification for that position. These would then be placed into a table for retrieval later.

#### Prereq files

- Genomic coordinates for all genes:  gene | chromosome | start | end

In [5]:
import pymysql
import csv

"""
Input file sample (with genes of interest or could be done with entire hg38 genome):

- SLFN12L
- ZNF506
- RBM4
- EIF3I
- NAA15
- cog3
"""

# --------
# Step 1: Read only bullet-pointed genes from file
# --------
def read_genes_from_file(filename):
    genes = []
    with open(filename, "r") as f:
        for line in f:
            line = line.strip()
            if line.startswith("- "):
                gene = line[2:].strip().upper()
                if gene:
                    genes.append(gene)
    return list(set(genes))  # deduplicate (case-insensitive)

genes = read_genes_from_file("/expanse/lustre/projects/csd933/aho2/RNA_Mod/Data/gene_list.txt")

# --------
# Step 2: Connect to UCSC
# --------
connection = pymysql.connect(host="genome-mysql.soe.ucsc.edu",
                             user="genome",
                             password="",
                             database="hg38")

cursor = connection.cursor()

query = """
SELECT g.name2, g.chrom, g.txStart, g.txEnd
FROM refGene g
WHERE UPPER(g.name2) = %s
"""

results = []
for gene in genes:
    cursor.execute(query, (gene,))
    rows = cursor.fetchall()
    if rows:
        # Pick longest transcript
        best = max(rows, key=lambda r: r[3] - r[2])
        start = best[2] + 1  # UCSC is 0-based, convert to 1-based
        end = best[3]
        results.append((best[0], best[1], start, end))
    else:
        results.append((gene, "not found", "-", "-"))

cursor.close()
connection.close()

# --------
# Step 3: Save results as proper table (CSV)
# --------
output_file = "/expanse/lustre/projects/csd933/aho2/RNA_Mod/Data/gene_locations.csv"
with open(output_file, "w", newline="") as f:
    writer = csv.writer(f)  # comma-separated by default
    writer.writerow(["gene", "chromosome", "start", "end"])
    for row in results:
        writer.writerow(row)

print(f"Results saved to {output_file}")

- Classification list: unnamed gene index | MR01-1 | MR01-2 | Category (Rows where Category starts with 'Primary' are processed.)

For this particular analysis, we analyzed all of the protein coding genes within the hg38 geneome. With our .bam files, we can filter the genes down based on how many reads they have (less than 20 reads).

In [ ]:
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import numpy as np
import os

genes_df = pd.read_csv("/expanse/lustre/projects/csd933/aho2/RNA_Mod/Data/hg38_all_gene_locations.csv")
samples = ["MR01-1", "MR01-2"]

all_sample_counts = {sample: {gene: 0 for gene in genes_df['gene']} for sample in samples}

for sample in samples:
    print(f"\n--- Processing Sample: {sample} ---")
    
    chromosomes = genes_df['chromosome'].unique()

    for chromosome in chromosomes:
        pickle_path = f"/expanse/lustre/projects/csd933/aho2/RNA_Mod/Data/{sample}/{sample}-tab/{sample}-tab-chr_pickles/{sample}-tab-{chromosome}.pkl"
        
        if not os.path.exists(pickle_path):
            continue

        print(f"  Mapping reads on {chromosome}...")
        
        with open(pickle_path, 'rb') as f:
            read_df = pickle.load(f)

        if read_df.empty:
            continue

        if 'reference_end' not in read_df.columns:
            read_df['reference_end'] = read_df['reference_start'] + read_df['query_sequence'].str.len()

        chrom_genes = genes_df[genes_df['chromosome'] == chromosome]

        for _, gene in chrom_genes.iterrows():
            mask = (read_df['reference_start'] < gene['end']) & (read_df['reference_end'] > gene['start'])
            all_sample_counts[sample][gene['gene']] += mask.sum()


df_counts = pd.DataFrame(all_sample_counts).fillna(0)
sample1, sample2 = samples[0], samples[1]
threshold = 10

is_s1_high = df_counts[sample1] >= threshold
is_s2_high = df_counts[sample2] >= threshold

# Apply buckets
conditions = [
    (is_s1_high & is_s2_high),                                # Both High
    (~is_s1_high & ~is_s2_high),                              # Both Low
    ((is_s1_high & ~is_s2_high) | (~is_s1_high & is_s2_high)) # Mixed
]
choices = ['Primary (Both >= 10)', 'Low (Both < 10)', 'Mixed (One < 10)']

df_counts['Category'] = np.select(conditions, choices, default='Unknown')

primary_genes = df_counts[df_counts['Category'] == 'Primary (Both >= 10)']
low_genes     = df_counts[df_counts['Category'] == 'Low (Both < 10)']
mixed_genes   = df_counts[df_counts['Category'] == 'Mixed (One < 10)']

print(f"\n{'='*60}")
print(f"GENE CLASSIFICATION REPORT (Threshold: {threshold} reads)")
print(f"{'='*60}")

for cat_name, data in [("PRIMARY", primary_genes), ("LOW", low_genes), ("MIXED", mixed_genes)]:
    print(f"\n>>> {cat_name} POOL (Total: {len(data)} genes)")
    if not data.empty:
        # Showing the first 10 genes as a preview
        print(data.head(10)[[sample1, sample2]].to_string())
        if len(data) > 10:
            print(f"... and {len(data)-10} more genes.")
    else:
        print("No genes found in this category.")

# Save the full list to a CSV for your records
df_counts.to_csv("gene_classification_master_list.csv")
print(f"\n[Success] Full table of {len(df_counts)} genes saved to 'gene_classification_master_list.csv'")

plt.figure(figsize=(10, 8))

# Log transformation for plotting (using +1 for 0-count genes)
log_s1 = np.log10(df_counts[sample1] + 1)
log_s2 = np.log10(df_counts[sample2] + 1)
log_thresh = np.log10(threshold)

colors = {'Primary (Both >= 10)': '#2ecc71', 
          'Low (Both < 10)': '#95a5a6', 
          'Mixed (One < 10)': '#e67e22'}

for cat in choices:
    mask = df_counts['Category'] == cat
    plt.scatter(log_s1[mask], log_s2[mask], alpha=0.5, label=cat, s=20, color=colors[cat])

plt.axvline(log_thresh, color='red', linestyle='--', alpha=0.6, label=f'Threshold ({threshold})')
plt.axhline(log_thresh, color='red', linestyle='--', alpha=0.6)

plt.title('Gene Read Counts: Cross-Sample Categorization', fontsize=14)
plt.xlabel(f'{sample1} - Log10(Reads + 1)', fontsize=12)
plt.ylabel(f'{sample2} - Log10(Reads + 1)', fontsize=12)
plt.legend(frameon=True, facecolor='white')
plt.grid(True, linestyle=':', alpha=0.5)

plt.tight_layout()
plt.savefig("gene_category_scatter.png")
plt.show()

#### Pipeline

In [ ]:
"""
Single-gene wrapper for Step 1 (extract).
Called by submit_pipeline.py via sbatch.

Usage:
    python process_single_gene_extract.py <gene> <chromosome> <start> <end> <sample>
"""

import sys
import pickle
import traceback
from pathlib import Path

# ── paste your extract.py helpers here, or import them ──────────────────────
# We import everything from extract.py directly so there's no duplication.
# Make sure extract.py is in the same directory or on PYTHONPATH.
sys.path.insert(0, str(Path(__file__).parent))

from extract import (
    DATA_DIR,
    SAMPLES,
    CHR_PICKLE_PATTERN,
    OUTPUT_DIR,
    _filter_alignments,
    process_gene_in_sample,
)

# ─────────────────────────────────────────────────────────────────────────────

def main():
    if len(sys.argv) != 6:
        print("Usage: python process_single_gene_extract.py <gene> <chrom> <start> <end> <sample>")
        sys.exit(1)

    gene       = sys.argv[1]
    chrom      = sys.argv[2]
    g_start    = int(sys.argv[3])
    g_end      = int(sys.argv[4])
    sample_label = sys.argv[5]

    if sample_label not in SAMPLES:
        print(f"ERROR: Unknown sample '{sample_label}'. Known: {list(SAMPLES.keys())}")
        sys.exit(1)

    sample_dir = SAMPLES[sample_label]

    print(f"[extract] Gene={gene}  Sample={sample_label}  {chrom}:{g_start}-{g_end}")

    out_path = OUTPUT_DIR / sample_label / f"{gene}.pkl"
    out_path.parent.mkdir(parents=True, exist_ok=True)

    pkl_path = CHR_PICKLE_PATTERN.format(
        sample_dir=sample_dir,
        sample=sample_label,
        chrom=chrom,
    )

    if not Path(pkl_path).exists():
        print(f"ERROR: chromosome pickle not found: {pkl_path}")
        sys.exit(1)

    print(f"  Loading {pkl_path} ...")
    with open(pkl_path, "rb") as fh:
        raw_df = pickle.load(fh)

    n_before = len(raw_df)
    raw_df = _filter_alignments(raw_df)
    n_dropped = n_before - len(raw_df)
    if n_dropped:
        print(f"  Dropped {n_dropped} secondary/supplementary alignments")

    try:
        summary_df = process_gene_in_sample(raw_df, chrom, g_start, g_end)
        with open(out_path, "wb") as fh:
            pickle.dump(summary_df, fh, protocol=pickle.HIGHEST_PROTOCOL)
        print(f"  -> {len(summary_df)} position-read rows saved to {out_path}")
    except Exception as exc:
        print(f"ERROR processing gene {gene}: {exc}")
        traceback.print_exc()
        sys.exit(1)


if __name__ == "__main__":
    main()

In [ ]:
"""
Extract per-position modification probabilities for each gene.

Input
-----
- hg38_all_gene_locations.csv      : columns  gene | chromosome | start | end
- gene_classification_master_list.csv : columns  <unnamed index>=gene | MR01-1 | MR01-2 | Category
                                        Only rows where Category starts with 'Primary' are used.
- Per-chromosome pickle files produced from BAM → DataFrame conversion.
  Each pickle is a pd.DataFrame with at minimum:
      query_sequence, cigarstring, reference_start, flag, MM (str), ML (str or array/list)

Output
------
- One pickle per gene × sample, saved under OUTPUT_DIR/{sample}/{gene}.pkl
  Each pickle is a pd.DataFrame with columns:
      Position (int, 0-based genomic) | Inosine | m6A | Unmod | Max
  where every row is one (read × position) pair contributing a call.
  Aggregation across reads is done in Step 2.

Fixes applied vs original:
  1. ML parsed from string repr via ast.literal_eval before np.asarray.
  2. Hard-clip (H) no longer advances read_pos (bases absent from query_sequence).
  3. ML scaled by 255 (SAM spec), not 256.
  4. Supplementary (flag & 2048) and secondary (flag & 256) alignments filtered out.
  5. None/nan query_sequence guard catches both "None" and "nan" string forms.
  6. Chromosome cache evicts entries when chromosome changes to bound memory.
  7. Reference span estimated from CIGAR for the overlap pre-filter.
"""

import ast
import re
import pickle
import traceback
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd

# ──────────────────────────────────────────────────────────────────────────────
# CONFIGURATION  – edit these paths before running
# ──────────────────────────────────────────────────────────────────────────────

DATA_DIR = "/expanse/lustre/projects/csd933/aho2/RNA_Mod/Data"

# Genomic coordinates for all genes:  gene | chromosome | start | end
GENE_LOCATIONS_CSV = f"{DATA_DIR}/hg38_all_gene_locations.csv"

# Classification list: unnamed gene index | MR01-1 | MR01-2 | Category
# Rows where Category starts with 'Primary' are processed.
CLASSIFICATION_CSV = f"{DATA_DIR}/gene_classification_master_list.csv"

# Sample definitions: {sample_label: directory_containing_per-chr_pickles}
SAMPLES = {
    "MR01-1": f"{DATA_DIR}/MR01-1/MR01-1-tab/MR01-1-tab-chr_pickles",
    "MR01-2": f"{DATA_DIR}/MR01-2/MR01-2-tab/MR01-2-tab-chr_pickles",
}

# Per-chromosome pickle filename pattern.
# {sample_dir}, {sample}, and {chrom} are substituted at runtime.
CHR_PICKLE_PATTERN = "{sample_dir}/{sample}-tab-{chrom}.pkl"

OUTPUT_DIR = Path(f"{DATA_DIR}/full_hg38/test")

# ──────────────────────────────────────────────────────────────────────────────
# CIGAR / MM-tag helpers
# ──────────────────────────────────────────────────────────────────────────────

_CIGAR_RE = re.compile(r"(\d+)([MIDNSHP=X])")


def _build_read_to_genome(cigar: str, ref_start: int) -> dict:
    """
    Return a dict {read_pos -> genome_pos} for every aligned (non-clipped)
    read position.

    SAM CIGAR consumption rules:
      M/=/X  – consume both query and reference
      I      – consume query only
      D/N    – consume reference only
      S      – consume query only  (bases ARE in query_sequence)
      H      – consume NEITHER     (bases are NOT in query_sequence)
      P      – consume neither (padding, rare)

    FIX: H must NOT advance read_pos because hard-clipped bases are absent
    from query_sequence entirely.  The original code incremented read_pos for
    both S and H, which shifted all downstream base-to-genome mappings for
    any hard-clipped read.
    """
    read_to_genome: dict[int, int] = {}
    read_pos = 0
    genome_pos = ref_start

    for length_str, op in _CIGAR_RE.findall(cigar):
        length = int(length_str)
        if op in ("M", "=", "X"):
            for _ in range(length):
                read_to_genome[read_pos] = genome_pos
                read_pos += 1
                genome_pos += 1
        elif op == "I":
            read_pos += length
        elif op in ("D", "N"):
            genome_pos += length
        elif op == "S":
            # Soft clip: bases present in query_sequence but not aligned
            read_pos += length
        elif op == "H":
            # FIX: Hard clip: bases NOT present in query_sequence – do not
            # advance read_pos.
            pass
        # P (padding) – skip

    return read_to_genome


def _cigar_reference_span(cigar: str) -> int:
    """
    Return the number of reference bases consumed by this CIGAR string.
    Used for the read-overlap pre-filter.
    Operators that consume reference: M, D, N, =, X.
    """
    span = 0
    for length_str, op in _CIGAR_RE.findall(cigar):
        if op in ("M", "D", "N", "=", "X"):
            span += int(length_str)
    return span


_MM_RE = re.compile(r"^([ACGTUN])([+-])([a-z0-9]+)([.?]?)((?:,\d+)*)$")


def _parse_mm_tag(mm_tag: str):
    """
    Parse one semicolon-separated MM sub-tag.

    Returns (base, strand, mod_codes, flag, deltas) where
      - mod_codes is a list of single-character or numeric code strings
      - flag is '.' (implicit unmod for unlisted positions), '?' (unknown),
        or '' (unspecified, treated as implicit)
      - deltas is a list of ints (skip counts between modified bases)
    """
    m = _MM_RE.match(mm_tag.strip())
    if not m:
        raise ValueError(f"Malformed MM sub-tag: {mm_tag!r}")

    base, strand, modcodes, flag, deltas_str = m.groups()

    if modcodes.isdigit():
        mod_codes = [modcodes]
    else:
        mod_codes = list(modcodes)

    deltas = [int(x) for x in deltas_str.strip(",").split(",") if x]
    return base, strand, mod_codes, flag, deltas


def _decode_positions(deltas: list, base_read_positions: list) -> list:
    """
    Given the skip-count deltas from an MM tag and the ordered list of
    *read* positions where `base` occurs (in the order they appear in the
    read sequence), return the read positions of modified bases.

    MM spec: delta[k] = number of occurrences of `base` to skip before the
    next modified base.  Occurrence index advances by delta[k]+1 each step.
    """
    occurrence_index = 0
    read_positions = []
    for delta in deltas:
        occurrence_index += delta
        if occurrence_index >= len(base_read_positions):
            break
        read_positions.append(base_read_positions[occurrence_index])
        occurrence_index += 1
    return read_positions


def _parse_ml(ml_val) -> np.ndarray:
    """
    Safely convert ML field to a float numpy array scaled to [0, 1].

    The BAM→DataFrame tabulation stores ML as a Python list repr string,
    e.g. '[4, 18, 15, 2, ...]'.  np.asarray on that string would iterate
    characters, producing nonsense.  We detect the string case and use
    ast.literal_eval to recover the actual list first.

    FIX: was dividing by 256; SAM spec mandates values in [0,255] mapped
    to probability via p = ML / 255.
    """
    if isinstance(ml_val, str):
        ml_val = ast.literal_eval(ml_val)
    arr = np.asarray(ml_val, dtype=float).flatten()
    return arr / 255.0


# ──────────────────────────────────────────────────────────────────────────────
# Alignment filtering
# ──────────────────────────────────────────────────────────────────────────────

# SAM flag bits
_FLAG_SECONDARY     = 0x100   # 256
_FLAG_SUPPLEMENTARY = 0x800   # 2048


def _filter_alignments(df: pd.DataFrame) -> pd.DataFrame:
    """
    Remove secondary and supplementary alignments.

    FIX: Secondary alignments (flag & 256) have no query_sequence stored.
    Supplementary alignments (flag & 2048) are chimeric split-read pieces;
    including them double-counts modifications for the same molecule.
    Only primary alignments (flag & 256 == 0 AND flag & 2048 == 0) are kept.
    """
    flags = df["flag"].astype(int)
    primary_mask = (
        ((flags & _FLAG_SECONDARY) == 0) &
        ((flags & _FLAG_SUPPLEMENTARY) == 0)
    )
    return df[primary_mask].copy()


# ──────────────────────────────────────────────────────────────────────────────
# Core per-read processing
# ──────────────────────────────────────────────────────────────────────────────

# Sentinel values that indicate a missing/invalid query_sequence after the
# BAM→DataFrame tabulation converts None to the string "None".
_INVALID_SEQ = frozenset({"nan", "None", "none", "", "NaN"})


def _process_one_read(row) -> list[tuple]:
    """
    Process a single read row.  Returns a list of
    (genomic_position, inosine_prob, m6a_prob, unmod_prob, max_call) tuples.
    """
    mm_val = row["MM"]
    ml_val = row["ML"]
    ref_start = int(row["reference_start"])
    seq = str(row["query_sequence"])
    cigar = str(row["cigarstring"])

    # FIX: str(None) == "None", not "nan" – guard against both forms.
    if (
        not mm_val
        or ml_val is None
        or not cigar
        or seq in _INVALID_SEQ
        or (isinstance(mm_val, float) and np.isnan(mm_val))
        or (isinstance(ml_val, float) and np.isnan(ml_val))
    ):
        return []

    raw_mm_tags = [t for t in str(mm_val).split(";") if t.strip()]

    try:
        ml_probs = _parse_ml(ml_val)
    except (ValueError, SyntaxError):
        return []

    read_to_genome = _build_read_to_genome(cigar, ref_start)

    # Build per-base ordered lists of *aligned* read positions only
    base_read_positions: dict[str, list[int]] = defaultdict(list)
    for read_pos in sorted(read_to_genome):
        base = seq[read_pos]
        base_read_positions[base].append(read_pos)

    ml_index = 0
    pos_mod_probs: dict[int, dict[str, float]] = defaultdict(dict)

    for mm_tag in raw_mm_tags:
        try:
            base, strand, mod_codes, flag, deltas = _parse_mm_tag(mm_tag)
        except ValueError:
            continue

        # '?' means probabilities are absent for unlisted positions – skip.
        if flag == "?":
            continue

        aligned_positions = base_read_positions.get(base, [])
        if not aligned_positions:
            continue

        modified_read_positions = _decode_positions(deltas, aligned_positions)
        n_mods = len(modified_read_positions)
        n_types = len(mod_codes)
        needed = n_mods * n_types

        if ml_index + needed > len(ml_probs):
            ml_index = len(ml_probs)
            continue

        probs_slice = ml_probs[ml_index: ml_index + needed]
        ml_index += needed

        for i_pos, rpos in enumerate(modified_read_positions):
            genome_pos = read_to_genome.get(rpos)
            if genome_pos is None:
                continue
            for i_mod, mod_code in enumerate(mod_codes):
                prob = float(probs_slice[i_pos * n_types + i_mod])
                existing = pos_mod_probs[genome_pos].get(mod_code, 0.0)
                pos_mod_probs[genome_pos][mod_code] = max(existing, prob)

    results = []
    for gpos, mod_dict in pos_mod_probs.items():
        inosine_p = mod_dict.get("17596", 0.0) or mod_dict.get("+17596", 0.0)
        m6a_p     = mod_dict.get("a", 0.0)     or mod_dict.get("+a", 0.0)
        unmod_p   = max(0.0, min(1.0, 1.0 - inosine_p - m6a_p))

        calls    = {"+17596": inosine_p, "+a": m6a_p, "Unmod": unmod_p}
        max_call = max(calls, key=calls.get)

        results.append((gpos, inosine_p, m6a_p, unmod_p, max_call))

    return results


def process_gene_in_sample(
    chr_df: pd.DataFrame,
    chrom: str,
    gene_start: int,
    gene_end: int,
    flanking: int = 0,
) -> pd.DataFrame:
    """
    Filter `chr_df` to reads overlapping [gene_start-flanking, gene_end+flanking],
    process MM/ML tags, and return a per-(read × position) DataFrame with columns:
        Position | Inosine | m6A | Unmod | Max

    FIX: overlap pre-filter now uses CIGAR-derived reference span instead of
    query_sequence length, so reads spanning large splice junctions are
    correctly included.
    """
    lo = gene_start - flanking
    hi = gene_end + flanking

    # FIX: compute reference span from CIGAR for an accurate overlap check.
    # Fall back to query_sequence length only when cigarstring is absent.
    if "cigarstring" in chr_df.columns:
        ref_spans = chr_df["cigarstring"].apply(
            lambda c: _cigar_reference_span(str(c)) if c and str(c) not in _INVALID_SEQ else 0
        )
    else:
        ref_spans = chr_df["query_sequence"].str.len().fillna(0)

    mask = (
        (chr_df["reference_start"] <= hi) &
        (chr_df["reference_start"] + ref_spans >= lo)
    )
    gene_df = chr_df[mask]

    rows = []
    errors = 0
    for _, row in gene_df.iterrows():
        try:
            rows.extend(_process_one_read(row))
        except Exception:
            errors += 1

    if errors:
        print(f"    Skipped {errors} reads due to parse errors")

    if not rows:
        return pd.DataFrame(columns=["Position", "Inosine", "m6A", "Unmod", "Max"])

    df = pd.DataFrame(rows, columns=["Position", "Inosine", "m6A", "Unmod", "Max"])
    df = df[(df["Position"] >= lo) & (df["Position"] <= hi)].copy()
    return df


# ──────────────────────────────────────────────────────────────────────────────
# Main loop
# ──────────────────────────────────────────────────────────────────────────────

def load_gene_table() -> pd.DataFrame:
    coords_df = pd.read_csv(GENE_LOCATIONS_CSV)
    coords_df = coords_df.rename(columns={coords_df.columns[0]: "gene"})

    class_df = pd.read_csv(CLASSIFICATION_CSV)
    class_df = class_df.rename(columns={class_df.columns[0]: "gene"})

    class_df = class_df[
        class_df["Category"].str.startswith("Primary", na=False)
    ][["gene"]].copy()

    gene_df = class_df.merge(coords_df, on="gene", how="inner")

    bad_mask = gene_df.apply(
        lambda r: r.astype(str).str.contains("not found", case=False).any(),
        axis=1,
    )
    gene_df = gene_df[~bad_mask].reset_index(drop=True)

    required = {"gene", "chromosome", "start", "end"}
    missing = required - set(gene_df.columns)
    if missing:
        raise ValueError(f"Missing expected columns after merge: {missing}")

    gene_df["start"] = gene_df["start"].astype(int)
    gene_df["end"]   = gene_df["end"].astype(int)

    print(f"  {len(gene_df)} Primary genes with valid coordinates found.")
    return gene_df


def main(single_gene: str = None):
    gene_df = load_gene_table()

    if single_gene is not None:
        gene_df = gene_df[gene_df["gene"] == single_gene].reset_index(drop=True)
        if gene_df.empty:
            print(f"ERROR: '{single_gene}' not found in Primary gene list or missing coordinates.")
            return
        print(f"TEST MODE: single gene '{single_gene}'\n")

    print(f"Processing {len(gene_df)} gene(s) across {len(SAMPLES)} samples.\n")

    # FIX: chromosome cache with eviction.
    # We track which chromosomes are loaded per sample and evict any entry
    # whose chromosome is no longer needed (genes are processed in coordinate
    # order so we won't revisit a chromosome once we move past it, assuming
    # the gene table is sorted by chromosome).  This bounds memory to at most
    # 2 × (number of samples) DataFrames at once instead of growing forever.
    chr_cache: dict[tuple, pd.DataFrame] = {}

    # Group genes by chromosome so we can evict cleanly
    gene_df_sorted = gene_df.sort_values(["chromosome", "start"]).reset_index(drop=True)
    chroms_in_order = list(gene_df_sorted["chromosome"].unique())
    processed = 0
    errors = 0
    prev_chrom = None

    for _, gene_row in gene_df_sorted.iterrows():
        gene   = gene_row["gene"]
        chrom  = str(gene_row["chromosome"])
        g_start = int(gene_row["start"])
        g_end   = int(gene_row["end"])

        # Evict cached chromosomes we've moved past
        if chrom != prev_chrom:
            keys_to_drop = [k for k in chr_cache if k[1] != chrom]
            for k in keys_to_drop:
                del chr_cache[k]
            prev_chrom = chrom

        print(f"{'='*60}")
        print(f"Gene: {gene}  ({chrom}:{g_start}-{g_end})")

        gene_ok = True
        for sample_label, sample_dir in SAMPLES.items():
            out_path = OUTPUT_DIR / sample_label / f"{gene}.pkl"
            out_path.parent.mkdir(parents=True, exist_ok=True)

            chr_key = (sample_label, chrom)
            if chr_key not in chr_cache:
                pkl_path = CHR_PICKLE_PATTERN.format(
                    sample_dir=sample_dir,
                    sample=sample_label,
                    chrom=chrom,
                )
                if not Path(pkl_path).exists():
                    print(f"  ERROR [{sample_label}]: chromosome pickle not found: {pkl_path}")
                    gene_ok = False
                    break
                print(f"  Loading {pkl_path} ...")
                with open(pkl_path, "rb") as fh:
                    raw_df = pickle.load(fh)

                # FIX: drop secondary and supplementary alignments immediately
                # after loading so every downstream operation works on primary
                # alignments only.
                n_before = len(raw_df)
                raw_df = _filter_alignments(raw_df)
                n_dropped = n_before - len(raw_df)
                if n_dropped:
                    print(f"    Dropped {n_dropped} secondary/supplementary alignments")

                chr_cache[chr_key] = raw_df

            chr_df = chr_cache[chr_key]
            try:
                summary_df = process_gene_in_sample(chr_df, chrom, g_start, g_end)
                with open(out_path, "wb") as fh:
                    pickle.dump(summary_df, fh, protocol=pickle.HIGHEST_PROTOCOL)
                print(f"  [{sample_label}] {len(summary_df)} position-read rows -> {out_path.name}")
            except Exception as exc:
                print(f"  ERROR [{sample_label}]: {exc}")
                traceback.print_exc()
                gene_ok = False

        if gene_ok:
            processed += 1
        else:
            errors += 1

    print(f"\n{'='*60}")
    print(f"Done.  Processed: {processed}  Errors: {errors}")


if __name__ == "__main__":
    import sys
    main(single_gene=sys.argv[1] if len(sys.argv) > 1 else None)

In [ ]:
"""
submit_extract.py - Step 1 submitter (one instance per sample).

Resumes automatically: if the output pickle for a gene already exists
and is non-empty, that gene is skipped.

Changes vs original:
  - ACCOUNT fixed to csd933.
  - Failed/timed-out jobs no longer kill the orchestrator.
  - Retry uses the next memory tier up, not a flat 64G ceiling.
  - Stats correctly distinguish skipped / submitted / sbatch-errors / retried.
  - sbatch account errors stop the run immediately (every job would fail).
  - OOM kills detected: COMPLETED-but-no-pickle treated as FAILED for retry.
  - LARGE_CHROMS expanded to include gene-dense chromosomes.

Usage:
    python submit_extract.py MR01-1
    python submit_extract.py MR01-2
"""

import os
import subprocess
import sys
import time
from pathlib import Path

import pandas as pd

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────

DRY_RUN       = False
LIMIT_TEST    = None
POLL_INTERVAL = 30

DATA_DIR      = "/expanse/lustre/projects/csd933/aho2/RNA_Mod/Data"
LOCATIONS_CSV = f"{DATA_DIR}/hg38_all_gene_locations.csv"
CLASS_CSV     = f"{DATA_DIR}/gene_classification_master_list.csv"

EXTRACT_OUT_DIR = Path(f"{DATA_DIR}/full_hg38/test")

SCRIPT_DIR    = os.path.dirname(os.path.abspath(__file__))

ACCOUNT       = "csd933"
PARTITION     = "shared"

# Expanded to include gene-dense chromosomes that need more memory
# despite being physically smaller than chr1-7.
LARGE_CHROMS  = {
    "chr1", "chr2", "chr3", "chr4", "chr5", "chr6", "chr7",
    "chr11", "chr17", "chr19", "chr22",
}

TERMINAL_STATES = {"COMPLETED", "FAILED", "CANCELLED", "TIMEOUT", "NODE_FAIL"}

# Memory tiers in order — retry steps up one tier from whatever was used
MEM_TIERS = ["8G", "12G", "16G", "20G", "24G", "32G", "48G", "64G"]

# ─────────────────────────────────────────────────────────────────────────────

if len(sys.argv) < 2:
    print("Usage: python submit_extract.py <sample_name>")
    sys.exit(1)

sample = sys.argv[1]

BASE_DIR   = f"{DATA_DIR}/full_hg38/pipeline_logs/extract/{sample}/"
LOG_DIR    = os.path.join(BASE_DIR, "logs/")
SBATCH_DIR = os.path.join(BASE_DIR, "sbatch_scripts/")
os.makedirs(LOG_DIR,    exist_ok=True)
os.makedirs(SBATCH_DIR, exist_ok=True)

# ── Load gene table ───────────────────────────────────────────────────────────

class_df = pd.read_csv(CLASS_CSV)
class_df.rename(columns={class_df.columns[0]: "gene"}, inplace=True)

loc_df = pd.read_csv(LOCATIONS_CSV)
loc_df = loc_df[
    ~loc_df.apply(lambda r: r.astype(str).str.contains("not found").any(), axis=1)
]

merged_df     = loc_df.merge(class_df[["gene", "Category"]], on="gene", how="inner")
primary_genes = merged_df[
    merged_df["Category"].str.contains("Primary", na=False, case=False)
].copy()

if LIMIT_TEST:
    print(f"!!! TEST MODE: first {LIMIT_TEST} genes only !!!")
    primary_genes = primary_genes.head(LIMIT_TEST)

total_genes = len(primary_genes)
print(f"[{sample}] Starting extract submission for {total_genes} genes...")

# ─────────────────────────────────────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────────────────────────────────────

def already_done(gene_name):
    out_pkl = EXTRACT_OUT_DIR / sample / f"{gene_name}.pkl"
    return out_pkl.exists() and out_pkl.stat().st_size > 0


def adaptive_mem(chromosome, gene_len):
    """Initial memory allocation based on chromosome and gene size."""
    if chromosome in LARGE_CHROMS:
        return (
            "16G" if gene_len < 5_000  else
            "20G" if gene_len < 20_000 else
            "24G" if gene_len < 50_000 else
            "32G"
        )
    return (
        "8G"  if gene_len < 5_000  else
        "12G" if gene_len < 20_000 else
        "16G" if gene_len < 50_000 else
        "24G"
    )


def next_mem_tier(current_mem):
    """
    Return the next memory tier above current_mem.
    If already at the ceiling, return None.
    """
    try:
        idx = MEM_TIERS.index(current_mem)
    except ValueError:
        current_gb = int(current_mem.replace("G", ""))
        for tier in MEM_TIERS:
            if int(tier.replace("G", "")) > current_gb:
                return tier
        return None

    if idx + 1 < len(MEM_TIERS):
        return MEM_TIERS[idx + 1]
    return None


def write_sbatch(gene_name, chromosome, start, end, mem, suffix=""):
    sbatch_file = os.path.join(SBATCH_DIR, f"{gene_name}{suffix}.sh")
    log_base    = os.path.join(LOG_DIR, f"{gene_name}{suffix}")

    with open(sbatch_file, "w") as f:
        f.write(f"""#!/bin/bash
#SBATCH --job-name="ext_{sample}_{gene_name}"
#SBATCH --output="{log_base}.%j.out"
#SBATCH --error="{log_base}.%j.err"
#SBATCH --partition={PARTITION}
#SBATCH --nodes=1
#SBATCH --ntasks-per-node=1
#SBATCH --cpus-per-task=4
#SBATCH --mem={mem}
#SBATCH -t 06:00:00
#SBATCH --account={ACCOUNT}
#SBATCH --requeue
#SBATCH --export=ALL

module reset
module load cpu/0.17.3b
module load gcc/10.2.0
source ~/.bashrc
source ~/miniconda3/etc/profile.d/conda.sh
conda activate rnaenv

python -u {SCRIPT_DIR}/process_single_gene_extract.py "{gene_name}" "{chromosome}" "{start}" "{end}" "{sample}"
""")
    return sbatch_file


def submit_and_wait(gene_name, chromosome, start, end, mem, index, suffix=""):
    """
    Submit a gene job and wait for completion.
    Returns (state, is_account_error).

    OOM detection: --requeue causes OOM-killed jobs to report COMPLETED
    to sacct even though the Python process was killed and no pickle was
    written. We detect this by checking for the output pickle after a
    COMPLETED state — if it's missing the job was silently OOM killed.
    """
    if DRY_RUN:
        print(f"[DRY RUN] [{sample}] [{index}/{total_genes}] "
              f"Would submit {gene_name} ({mem})")
        return "COMPLETED", False

    sbatch_file = write_sbatch(gene_name, chromosome, start, end, mem, suffix)

    result = subprocess.run(
        ["sbatch", "--parsable", sbatch_file],
        capture_output=True, text=True,
    )

    if result.returncode != 0:
        err = result.stderr.strip()
        print(f"[{sample}] ERROR submitting {gene_name}: {err}")
        is_account_error = (
            "balance" in err.lower() or
            "policy" in err.lower() or
            "qos"     in err.lower()
        )
        return "SBATCH_FAILED", is_account_error

    job_id = result.stdout.strip()
    print(f"[{sample}] [{index}/{total_genes}] Submitted {gene_name} "
          f"(Job {job_id}, mem={mem}). Waiting...")

    not_visible_count = 0

    while True:
        time.sleep(POLL_INTERVAL)

        q = subprocess.run(
            ["squeue", "-h", "-j", job_id],
            capture_output=True, text=True,
        )
        if job_id in q.stdout:
            print(f"[{sample}] Job {job_id} ({gene_name}) still in queue...")
            not_visible_count = 0
            continue

        a = subprocess.run(
            ["sacct", "-j", job_id, "--format=State", "--noheader", "-P"],
            capture_output=True, text=True,
        )
        states = {s.strip().split()[0] for s in a.stdout.strip().splitlines() if s.strip()}
        terminal = states & TERMINAL_STATES

        if terminal:
            state = next(iter(terminal))
            print(f"[{sample}] Job {job_id} ({gene_name}) finished: {state}\n")

            # ── OOM detection ─────────────────────────────────────────────────
            # --requeue makes OOM-killed jobs report COMPLETED to sacct.
            # Check that the pickle actually exists — if not, the process
            # was killed before writing output and needs a retry with more memory.
            if state == "COMPLETED" and not already_done(gene_name):
                print(f"[{sample}] WARNING: {gene_name} reported COMPLETED "
                      f"but pickle is missing — likely OOM killed. "
                      f"Treating as FAILED for retry.")
                return "FAILED", False

            return state, False

        not_visible_count += 1
        print(f"[{sample}] Job {job_id} not yet visible "
              f"(check #{not_visible_count})...")

        if not_visible_count > 60:
            print(f"[{sample}] WARNING: Job {job_id} never appeared after 10 min.")
            return "TIMEOUT", False


# ─────────────────────────────────────────────────────────────────────────────
# First pass
# ─────────────────────────────────────────────────────────────────────────────

submitted      = 0
skipped        = 0
account_errors = 0
failed         = []   # list of (i, row, failed_mem) for retry

print(f"\n[{sample}] === FIRST PASS ===\n")

for i, (idx, row) in enumerate(primary_genes.iterrows(), 1):
    gene_name  = row["gene"]
    chromosome = str(row["chromosome"])
    start      = int(row["start"])
    end        = int(row["end"])
    gene_len   = end - start

    if already_done(gene_name):
        print(f"[{sample}] [{i}/{total_genes}] SKIPPING {gene_name} (already done)")
        skipped += 1
        continue

    mem            = adaptive_mem(chromosome, gene_len)
    state, is_acct = submit_and_wait(gene_name, chromosome, start, end, mem, i)

    if state == "COMPLETED":
        submitted += 1

    elif state == "SBATCH_FAILED":
        if is_acct:
            account_errors += 1
            print(f"[{sample}] FATAL: account/QOS error. "
                  f"Fix account balance and resubmit.")
            print(f"[{sample}] Progress: submitted={submitted} "
                  f"skipped={skipped} account_errors={account_errors}")
            sys.exit(1)
        else:
            print(f"[{sample}] sbatch error for {gene_name} — queued for retry.")
            failed.append((i, row, mem))

    else:
        # FAILED, TIMEOUT, or OOM-detected FAILED
        print(f"[{sample}] {gene_name} ended with {state} (mem={mem}) "
              f"— queued for retry.")
        failed.append((i, row, mem))

# ─────────────────────────────────────────────────────────────────────────────
# Retry pass — one tier up from whatever memory was used originally
# ─────────────────────────────────────────────────────────────────────────────

retry_submitted = 0
retry_failed    = []

if failed:
    print(f"\n[{sample}] === RETRY PASS: {len(failed)} genes ===\n")

    for i, row, original_mem in failed:
        gene_name  = row["gene"]
        chromosome = str(row["chromosome"])
        start      = int(row["start"])
        end        = int(row["end"])

        if already_done(gene_name):
            print(f"[{sample}] SKIPPING {gene_name} (appeared during first pass)")
            retry_submitted += 1
            continue

        retry_mem = next_mem_tier(original_mem)
        if retry_mem is None:
            print(f"[{sample}] {gene_name} already at memory ceiling "
                  f"({original_mem}) — cannot retry higher.")
            retry_failed.append((gene_name, original_mem, "AT_CEILING"))
            continue

        print(f"[{sample}] Retrying {gene_name}: {original_mem} → {retry_mem}")

        state, is_acct = submit_and_wait(
            gene_name, chromosome, start, end,
            retry_mem, i, suffix="_retry"
        )

        if state == "COMPLETED":
            retry_submitted += 1

        elif state == "SBATCH_FAILED":
            if is_acct:
                account_errors += 1
                print(f"[{sample}] FATAL: account/QOS error on retry.")
                sys.exit(1)
            else:
                retry_failed.append((gene_name, retry_mem, state))

        else:
            print(f"[{sample}] {gene_name} failed again on retry "
                  f"(mem={retry_mem}): {state}")
            retry_failed.append((gene_name, retry_mem, state))

# ─────────────────────────────────────────────────────────────────────────────
# Final summary
# ─────────────────────────────────────────────────────────────────────────────

total_done = skipped + submitted + retry_submitted

print(f"\n[{sample}] ============================================")
print(f"[{sample}] Done.")
print(f"[{sample}]   Total Primary genes:      {total_genes}")
print(f"[{sample}]   Already done (skipped):   {skipped}")
print(f"[{sample}]   Submitted (first pass):   {submitted}")
print(f"[{sample}]   Queued for retry:         {len(failed)}")
print(f"[{sample}]   Retry succeeded:          {retry_submitted}")
print(f"[{sample}]   Retry failed:             {len(retry_failed)}")
print(f"[{sample}]   Account/sbatch errors:    {account_errors}")
print(f"[{sample}]   Total with pickles now:   {total_done}")
print(f"[{sample}]   Still missing:            {total_genes - total_done}")

if retry_failed:
    print(f"\n[{sample}] Genes that failed even after retry:")
    for gene, mem, state in retry_failed:
        print(f"  {gene}: failed at {mem} with {state}")

print(f"[{sample}] ============================================")

In [ ]:
#!/bin/bash
# Submits the extract submitters as their own SLURM jobs so they survive
# notebook/session closure completely.

source ~/miniconda3/etc/profile.d/conda.sh
conda activate rnaenv

SCRIPT_DIR="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)"
DATA_DIR="/expanse/lustre/projects/csd933/aho2/RNA_Mod/Data"
LOG_DIR="$DATA_DIR/full_hg38/pipeline_logs"
mkdir -p "$LOG_DIR"

# ── Submit MR01-1 extractor as a SLURM job ───────────────────────────────────
cat > "$LOG_DIR/run_extract_MR01-1.sh" << 'EOF'
#!/bin/bash
#SBATCH --job-name="pipeline_MR01-1"
#SBATCH --output="/expanse/lustre/projects/csd933/aho2/RNA_Mod/Data/full_hg38/pipeline_logs/extract_MR01-1.log"
#SBATCH --error="/expanse/lustre/projects/csd933/aho2/RNA_Mod/Data/full_hg38/pipeline_logs/extract_MR01-1.err"
#SBATCH --partition=shared
#SBATCH --nodes=1
#SBATCH --ntasks-per-node=1
#SBATCH --cpus-per-task=1
#SBATCH --mem=4G
#SBATCH -t 48:00:00
#SBATCH --account=csd933
#SBATCH --export=ALL

module reset
module load cpu/0.17.3b
module load gcc/10.2.0
source ~/.bashrc
source ~/miniconda3/etc/profile.d/conda.sh
conda activate rnaenv

python -u SCRIPT_DIR_PLACEHOLDER/submit_extract.py MR01-1
EOF

# # ── Submit MR01-2 extractor as a SLURM job ───────────────────────────────────
# cat > "$LOG_DIR/run_extract_MR01-2.sh" << 'EOF'
# #!/bin/bash
# #SBATCH --job-name="pipeline_MR01-2"
# #SBATCH --output="/expanse/lustre/projects/csd933/aho2/RNA_Mod/Data/full_hg38/pipeline_logs/extract_MR01-2.log"
# #SBATCH --error="/expanse/lustre/projects/csd933/aho2/RNA_Mod/Data/full_hg38/pipeline_logs/extract_MR01-2.err"
# #SBATCH --partition=shared
# #SBATCH --nodes=1
# #SBATCH --ntasks-per-node=1
# #SBATCH --cpus-per-task=1
# #SBATCH --mem=4G
# #SBATCH -t 48:00:00
# #SBATCH --account=csd922
# #SBATCH --export=ALL

# module reset
# module load cpu/0.17.3b
# module load gcc/10.2.0
# source ~/.bashrc
# source ~/miniconda3/etc/profile.d/conda.sh
# conda activate rnaenv

# python -u SCRIPT_DIR_PLACEHOLDER/submit_extract.py MR01-2
# EOF

# Replace placeholder with actual script dir
sed -i "s|SCRIPT_DIR_PLACEHOLDER|$SCRIPT_DIR|g" "$LOG_DIR/run_extract_MR01-1.sh"
# sed -i "s|SCRIPT_DIR_PLACEHOLDER|$SCRIPT_DIR|g" "$LOG_DIR/run_extract_MR01-2.sh"

# Submit both
JOB1=$(sbatch --parsable "$LOG_DIR/run_extract_MR01-1.sh")
# JOB2=$(sbatch --parsable "$LOG_DIR/run_extract_MR01-2.sh")

echo "============================================"
echo "  Pipeline submitted as SLURM jobs"
echo "  MR01-1 orchestrator: Job $JOB1"
# echo "  MR01-2 orchestrator: Job $JOB2"
echo ""
echo "  These jobs will survive notebook closure."
echo ""
echo "  Track logs:"
echo "    tail -f $LOG_DIR/extract_MR01-1.log"
# echo "    tail -f $LOG_DIR/extract_MR01-2.log"
echo ""
echo "  Check status:"
echo "    squeue -u $USER"
echo "============================================"

### Annotate genomic positions with transcript features and aggregate

Now that we have the filtered genes with valid readings, we want to create our aggregated output table for the regions of interest (Exonic, Introic, UTR 3', UTR 5') for both of our samples (MR01-1 and MR01-2). Within these tables, we retrieved counts per kilobase, means, and probabilities for each of the regions for each of the samples. 

Sample output for gene AZIN1:

#### Pipeline

In [ ]:
"""
Single-gene wrapper for Step 2 (output/aggregate).
Called by submit_pipeline.py via sbatch.

Usage:
    python process_single_gene_output.py <gene> <chromosome> <start> <end>

Note: output.py aggregates BOTH samples together for a given gene,
so this script does not take a sample argument.
"""

import sys
import pickle
import traceback
from pathlib import Path

sys.path.insert(0, str(Path(__file__).parent))

from output import (
    DATA_DIR,
    SAMPLES,
    INPUT_DIR,
    OUTPUT_DIR,
    BED_PATH,
    BED_FLANKING,
    load_bed_region,
    build_feature_trees,
    compute_feature_lengths,
    classify_positions,
    aggregate,
)

# ─────────────────────────────────────────────────────────────────────────────

def main():
    if len(sys.argv) != 5:
        print("Usage: python process_single_gene_output.py <gene> <chrom> <start> <end>")
        sys.exit(1)

    gene    = sys.argv[1]
    chrom   = sys.argv[2]
    g_start = int(sys.argv[3])
    g_end   = int(sys.argv[4])

    print(f"[output] Gene={gene}  {chrom}:{g_start}-{g_end}")

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    # Load per-sample pickles written by extract step
    sample_dfs = {}
    for sample in SAMPLES:
        pkl_path = INPUT_DIR / sample / f"{gene}.pkl"
        if not pkl_path.exists():
            print(f"ERROR: missing extract output {pkl_path}")
            sys.exit(1)
        with open(pkl_path, "rb") as fh:
            sample_dfs[sample] = pickle.load(fh)
        print(f"  [{sample}] {len(sample_dfs[sample])} rows loaded")

    bed_df = load_bed_region(BED_PATH, chrom, g_start, g_end, BED_FLANKING)
    if bed_df.empty:
        print(f"WARNING: no BED transcripts found near {gene} — skipping")
        sys.exit(1)

    trees           = build_feature_trees(bed_df)
    feature_lengths = compute_feature_lengths(trees)
    print("  Feature lengths (bp): " +
          ", ".join(f"{f}={feature_lengths[f]:,}" for f in ["UTR_5","Exon","UTR_3","Intron"]))

    all_positions: set[int] = set()
    for df in sample_dfs.values():
        all_positions |= set(df["Position"].dropna().astype(int))

    pos_to_feature = classify_positions(all_positions, trees)
    coverage = len(pos_to_feature) / max(len(all_positions), 1) * 100
    print(f"  {len(pos_to_feature)}/{len(all_positions)} positions mapped ({coverage:.1f}%)")

    try:
        result = aggregate(sample_dfs, pos_to_feature, feature_lengths, gene)

        out_pkl = OUTPUT_DIR / f"{gene}.pkl"
        with open(out_pkl, "wb") as fh:
            pickle.dump(result, fh, protocol=pickle.HIGHEST_PROTOCOL)

        tsv_path = out_pkl.with_suffix(".tsv")
        result.to_csv(tsv_path, sep="\t", index=False)

        print(f"  -> Saved {out_pkl.name}  ({len(result)} rows)")
    except Exception as exc:
        print(f"ERROR aggregating gene {gene}: {exc}")
        traceback.print_exc()
        sys.exit(1)


if __name__ == "__main__":
    main()

In [ ]:
"""
Annotate genomic positions with transcript features and aggregate.

Reads the per-(gene × sample) pickles produced by Step 1, maps every
genomic position to its feature (UTR_5 / Exon / UTR_3 / Intron), then
produces a summary table per gene.
"""

import pickle
import traceback
from pathlib import Path

import numpy as np
import pandas as pd
from intervaltree import IntervalTree

# ──────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ──────────────────────────────────────────────────────────────────────────────

DATA_DIR = "/expanse/lustre/projects/csd933/aho2/RNA_Mod/Data"

GENE_LOCATIONS_CSV = f"{DATA_DIR}/hg38_all_gene_locations.csv"
BED_PATH           = f"{DATA_DIR}/bed/ucscRefSeq.bed"

SAMPLES = ["MR01-1", "MR01-2"]

INPUT_DIR  = Path(f"{DATA_DIR}/full_hg38/test")
OUTPUT_DIR = Path(f"{DATA_DIR}/full_hg38/test/processed_gene")

BED_FLANKING = 5_000

DISPLAY_ORDER  = ["UTR_5", "Exon", "UTR_3", "Intron"]
PRIORITY_ORDER = ["Exon", "UTR_5", "UTR_3", "Intron"]

MOD_ORDER = ["Unmod", "m6A", "Inosine"]

_MAX_TO_MOD = {
    "+a":     "m6A",
    "a":      "m6A",
    "+17596": "Inosine",
    "17596":  "Inosine",
    "Unmod":  "Unmod",
    "unmod":  "Unmod",
}

# ──────────────────────────────────────────────────────────────────────────────
# BED helpers
# ──────────────────────────────────────────────────────────────────────────────

_BED_COLS = [
    "chrom", "txStart", "txEnd", "name", "score", "strand",
    "thickStart", "thickEnd", "itemRgb", "blockCount", "blockSizes", "blockStarts",
]


def load_bed_region(
    bed_path: str, chrom: str, start: int, end: int, flanking: int = 5_000
) -> pd.DataFrame:
    df = pd.read_csv(
        bed_path, sep="\t", header=None, names=_BED_COLS,
        usecols=range(12), na_filter=False, dtype=str,
    )
    numeric = ["txStart", "txEnd", "thickStart", "thickEnd", "blockCount"]
    for col in numeric:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.dropna(subset=numeric)

    lo, hi = start - flanking, end + flanking
    df = df[
        (df["chrom"] == chrom)
        & (df["txEnd"]   >= lo)
        & (df["txStart"] <= hi)
    ].copy()

    df = df.drop_duplicates(subset=["blockSizes", "blockStarts"]).reset_index(drop=True)
    return df


def _parse_exons(row) -> list[tuple[int, int]]:
    starts   = [int(x) for x in str(row["blockStarts"]).rstrip(",").split(",") if x]
    sizes    = [int(x) for x in str(row["blockSizes"]).rstrip(",").split(",") if x]
    tx_start = int(row["txStart"])
    return [(tx_start + s, tx_start + s + sz) for s, sz in zip(starts, sizes)]


def build_feature_trees(bed_df: pd.DataFrame) -> dict[str, IntervalTree]:
    trees: dict[str, IntervalTree] = {f: IntervalTree() for f in DISPLAY_ORDER}

    for _, row in bed_df.iterrows():
        exons = _parse_exons(row)
        if not exons:
            continue

        thick_start = int(row["thickStart"])
        thick_end   = int(row["thickEnd"])
        strand      = str(row["strand"])

        def add(feature: str, s: int, e: int):
            if e > s:
                trees[feature][s:e] = True

        for exon_s, exon_e in exons:
            if exon_e <= thick_start:
                feat = "UTR_5" if strand == "+" else "UTR_3"
                add(feat, exon_s, exon_e)
            elif exon_s >= thick_end:
                feat = "UTR_3" if strand == "+" else "UTR_5"
                add(feat, exon_s, exon_e)
            else:
                if exon_s < thick_start:
                    feat = "UTR_5" if strand == "+" else "UTR_3"
                    add(feat, exon_s, thick_start)
                cds_s = max(exon_s, thick_start)
                cds_e = min(exon_e, thick_end)
                add("Exon", cds_s, cds_e)
                if exon_e > thick_end:
                    feat = "UTR_3" if strand == "+" else "UTR_5"
                    add(feat, thick_end, exon_e)

        for i in range(len(exons) - 1):
            intron_s = exons[i][1]
            intron_e = exons[i + 1][0]
            add("Intron", intron_s, intron_e)

    return trees


def _merge_tree_length(tree: IntervalTree) -> int:
    if not tree:
        return 0
    ivs = sorted((iv.begin, iv.end) for iv in tree)
    merged_len = 0
    cur_s, cur_e = ivs[0]
    for s, e in ivs[1:]:
        if s <= cur_e:
            cur_e = max(cur_e, e)
        else:
            merged_len += cur_e - cur_s
            cur_s, cur_e = s, e
    merged_len += cur_e - cur_s
    return merged_len


def compute_feature_lengths(trees: dict[str, IntervalTree]) -> dict[str, int]:
    return {feat: _merge_tree_length(trees[feat]) for feat in DISPLAY_ORDER}


# ──────────────────────────────────────────────────────────────────────────────
# Position classification
# ──────────────────────────────────────────────────────────────────────────────

def classify_positions(
    positions: set[int],
    trees: dict[str, IntervalTree],
) -> dict[int, str]:
    pos_to_feature: dict[int, str] = {}
    for pos in positions:
        for feat in PRIORITY_ORDER:
            if trees[feat].overlaps(pos, pos + 1):
                pos_to_feature[pos] = feat
                break
    return pos_to_feature


# ──────────────────────────────────────────────────────────────────────────────
# Aggregation
# ──────────────────────────────────────────────────────────────────────────────

def aggregate(
    sample_dfs: dict[str, pd.DataFrame],
    pos_to_feature: dict[int, str],
    feature_lengths: dict[str, int],
    gene: str,
) -> pd.DataFrame:
    idx = pd.MultiIndex.from_product(
        [DISPLAY_ORDER, MOD_ORDER], names=["Feature", "Modification"]
    )
    result = pd.DataFrame(index=idx).reset_index()

    for sample, df in sample_dfs.items():
        if df.empty:
            result[f"Count_{sample}"] = 0
            result[f"CPK_{sample}"]   = np.nan
            result[f"{sample}"]       = "0.000000 ± NaN"
            continue

        df = df.copy()
        df["Feature"] = df["Position"].map(pos_to_feature)
        df = df.dropna(subset=["Feature"])

        df["Mod_canonical"] = df["Max"].map(_MAX_TO_MOD)
        counts = (
            df.dropna(subset=["Mod_canonical"])
            .groupby(["Feature", "Mod_canonical"])
            .size()
            .rename("Count")
            .reset_index()
            .rename(columns={"Mod_canonical": "Modification"})
        )

        prob_long = df.melt(
            id_vars=["Position", "Feature"],
            value_vars=["Inosine", "m6A", "Unmod"],
            var_name="Modification",
            value_name="Prob",
        )
        stats = (
            prob_long.groupby(["Feature", "Modification"])["Prob"]
            .agg(["mean", "std"])
            .reset_index()
        )

        counts = (
            counts.set_index(["Feature", "Modification"])
            .reindex(idx, fill_value=0)
            .reset_index()
        )
        stats = (
            stats.set_index(["Feature", "Modification"])
            .reindex(idx)
            .reset_index()
        )

        result[f"Count_{sample}"] = counts["Count"].values

        feat_len_map = {f: feature_lengths.get(f, 0) for f in DISPLAY_ORDER}
        cpk_values = []
        for _, row in counts.iterrows():
            fl = feat_len_map.get(row["Feature"], 0)
            cpk_values.append(
                round(row["Count"] / (fl / 1000.0), 6) if fl > 0 else np.nan
            )
        result[f"CPK_{sample}"] = cpk_values

        mean_std = []
        for _, row in stats.iterrows():
            m     = row["mean"] if pd.notna(row["mean"]) else 0.0
            s_str = f"{row['std']:.6f}" if pd.notna(row["std"]) else "NaN"
            mean_std.append(f"{m:.6f} ± {s_str}")
        result[f"{sample}"] = mean_std

    result.insert(0, "Gene", gene)

    feat_cat = pd.CategoricalDtype(DISPLAY_ORDER, ordered=True)
    mod_cat  = pd.CategoricalDtype(MOD_ORDER,     ordered=True)
    result["Feature"]      = result["Feature"].astype(feat_cat)
    result["Modification"] = result["Modification"].astype(mod_cat)
    result = result.sort_values(["Feature", "Modification"]).reset_index(drop=True)

    return result


# ──────────────────────────────────────────────────────────────────────────────
# Main — for direct invocation only, not used by the pipeline
# ──────────────────────────────────────────────────────────────────────────────

def main(single_gene: str = None):
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    # Build gene list from intersection of extract folders
    genes_per_sample = {}
    for sample in SAMPLES:
        sample_dir = INPUT_DIR / sample
        genes_per_sample[sample] = {p.stem for p in sample_dir.glob("*.pkl")}

    all_genes = set.intersection(*[genes_per_sample[s] for s in SAMPLES])

    loc_df = pd.read_csv(GENE_LOCATIONS_CSV)
    loc_df.rename(columns={loc_df.columns[0]: "gene"}, inplace=True)
    loc_df = loc_df[
        ~loc_df.apply(lambda r: r.astype(str).str.contains("not found", case=False).any(), axis=1)
    ].reset_index(drop=True)
    coord_map = loc_df.set_index("gene")[["chromosome", "start", "end"]].to_dict("index")

    if single_gene is not None:
        if single_gene not in all_genes:
            print(f"ERROR: '{single_gene}' not found in intersection of extract folders.")
            return
        if single_gene not in coord_map:
            print(f"ERROR: '{single_gene}' has no coordinates in {GENE_LOCATIONS_CSV}.")
            return
        gene_list = [single_gene]
        print(f"TEST MODE: single gene '{single_gene}'\n")
    else:
        gene_list = sorted(g for g in all_genes if g in coord_map)

    print(f"Aggregating {len(gene_list)} gene(s).\n")
    processed = errors = 0

    for gene in gene_list:
        c       = coord_map[gene]
        chrom   = str(c["chromosome"])
        g_start = int(c["start"])
        g_end   = int(c["end"])

        print(f"{'='*60}")
        print(f"Gene: {gene}  ({chrom}:{g_start}-{g_end})")

        try:
            sample_dfs = {}
            for sample in SAMPLES:
                pkl_path = INPUT_DIR / sample / f"{gene}.pkl"
                with open(pkl_path, "rb") as fh:
                    sample_dfs[sample] = pickle.load(fh)
                print(f"  [{sample}] {len(sample_dfs[sample])} rows loaded")

            bed_df = load_bed_region(BED_PATH, chrom, g_start, g_end, BED_FLANKING)
            if bed_df.empty:
                print(f"  WARNING: no BED transcripts found — skipping")
                errors += 1
                continue

            trees           = build_feature_trees(bed_df)
            feature_lengths = compute_feature_lengths(trees)
            print("  Feature lengths (bp): " +
                  ", ".join(f"{f}={feature_lengths[f]:,}" for f in DISPLAY_ORDER))

            all_positions: set[int] = set()
            for df in sample_dfs.values():
                all_positions |= set(df["Position"].dropna().astype(int))

            pos_to_feature = classify_positions(all_positions, trees)
            coverage = len(pos_to_feature) / max(len(all_positions), 1) * 100
            print(f"  {len(pos_to_feature)}/{len(all_positions)} positions mapped ({coverage:.1f}%)")

            result = aggregate(sample_dfs, pos_to_feature, feature_lengths, gene)

            out_path = OUTPUT_DIR / f"{gene}.pkl"
            with open(out_path, "wb") as fh:
                pickle.dump(result, fh, protocol=pickle.HIGHEST_PROTOCOL)

            tsv_path = out_path.with_suffix(".tsv")
            result.to_csv(tsv_path, sep="\t", index=False)

            print(f"  Saved {out_path.name}  ({len(result)} rows)")
            processed += 1

        except Exception as exc:
            print(f"  ERROR: {exc}")
            traceback.print_exc()
            errors += 1

    print(f"\n{'='*60}")
    print(f"Done.  Processed: {processed}  Errors: {errors}")
    print(f"Output: {OUTPUT_DIR}")


if __name__ == "__main__":
    import sys
    main(single_gene=sys.argv[1] if len(sys.argv) > 1 else None)

In [ ]:
"""
submit_output.py – Step 2 submitter (single instance, runs after extract).

Gene list is derived from the INTERSECTION of all sample extract folders —
only genes present in every sample are processed. Coordinates are pulled
from hg38_all_gene_locations.csv.

Set DRY_RUN = True to test without submitting any jobs.
Set LIMIT_TEST = N to process only the first N genes.
"""

import os
import subprocess
import time
from pathlib import Path

import pandas as pd

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────

DRY_RUN         = False    # ← set False when ready to run for real
LIMIT_TEST      = None       # ← set None to process all genes
POLL_INTERVAL   = 15

DATA_DIR        = "/expanse/lustre/projects/csd933/aho2/RNA_Mod/Data"
LOCATIONS_CSV   = f"{DATA_DIR}/hg38_all_gene_locations.csv"

SAMPLES         = ["MR01-1", "MR01-2"]

EXTRACT_OUT_DIR = Path(f"{DATA_DIR}/full_hg38/test")
OUTPUT_DIR      = Path(f"{DATA_DIR}/full_hg38/test/processed_gene")

SCRIPT_DIR      = os.path.dirname(os.path.abspath(__file__))

ACCOUNT         = "csd922"
PARTITION       = "shared"

BASE_DIR        = f"{DATA_DIR}/full_hg38/pipeline_logs/output/"
LOG_DIR         = os.path.join(BASE_DIR, "logs/")
SBATCH_DIR      = os.path.join(BASE_DIR, "sbatch_scripts/")
os.makedirs(LOG_DIR,    exist_ok=True)
os.makedirs(SBATCH_DIR, exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────────
# Build gene list — intersection of all sample extract folders
# ─────────────────────────────────────────────────────────────────────────────

genes_per_sample = {}
for sample in SAMPLES:
    sample_dir = EXTRACT_OUT_DIR / sample
    genes_per_sample[sample] = {p.stem for p in sample_dir.glob("*.pkl")}
    print(f"[output] {sample}: {len(genes_per_sample[sample])} extract pickles found")

# Intersection — only genes present in ALL samples
all_extracted = set.intersection(*[genes_per_sample[s] for s in SAMPLES])
print(f"[output] Genes present in ALL samples (intersection): {len(all_extracted)}")

# Report genes missing from any sample
for sample in SAMPLES:
    only_here = genes_per_sample[sample] - all_extracted
    if only_here:
        other = [s for s in SAMPLES if s != sample]
        print(f"[output] WARNING: {len(only_here)} genes only in {sample}, "
              f"missing from {other} — will be skipped.")

# ─────────────────────────────────────────────────────────────────────────────
# Load coordinates
# ─────────────────────────────────────────────────────────────────────────────

loc_df = pd.read_csv(LOCATIONS_CSV)
loc_df.rename(columns={loc_df.columns[0]: "gene"}, inplace=True)
loc_df = loc_df[
    ~loc_df.apply(lambda r: r.astype(str).str.contains("not found", case=False).any(), axis=1)
].reset_index(drop=True)

coord_map = loc_df.set_index("gene")[["chromosome", "start", "end"]].to_dict("index")

# ─────────────────────────────────────────────────────────────────────────────
# Build final gene table
# ─────────────────────────────────────────────────────────────────────────────

gene_rows      = []
missing_coords = []

for gene in sorted(all_extracted):
    if gene in coord_map:
        c = coord_map[gene]
        gene_rows.append({
            "gene":       gene,
            "chromosome": c["chromosome"],
            "start":      int(c["start"]),
            "end":        int(c["end"]),
        })
    else:
        missing_coords.append(gene)

if missing_coords:
    print(f"[output] WARNING: {len(missing_coords)} genes have no coordinates "
          f"and will be skipped:")
    for g in missing_coords:
        print(f"  {g}")

all_genes = pd.DataFrame(gene_rows)

if LIMIT_TEST:
    print(f"\n!!! TEST MODE: first {LIMIT_TEST} genes only !!!")
    all_genes = all_genes.head(LIMIT_TEST)

total = len(all_genes)
print(f"[output] Will process {total} genes.\n")

if DRY_RUN:
    print("=" * 60)
    print("  DRY RUN — no jobs will be submitted")
    print("  Set DRY_RUN = False to run for real")
    print("=" * 60 + "\n")

# ─────────────────────────────────────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────────────────────────────────────

def already_done(gene_name):
    out_pkl = OUTPUT_DIR / f"{gene_name}.pkl"
    return out_pkl.exists() and out_pkl.stat().st_size > 0


def submit_gene(gene_name, chromosome, start, end, gene_index):
    sbatch_file = os.path.join(SBATCH_DIR, f"{gene_name}.sh")
    log_base    = os.path.join(LOG_DIR, f"{gene_name}")

    with open(sbatch_file, "w") as f:
        f.write(f"""#!/bin/bash
#SBATCH --job-name="out_{gene_name}"
#SBATCH --output="{log_base}.%j.out"
#SBATCH --error="{log_base}.%j.err"
#SBATCH --partition={PARTITION}
#SBATCH --nodes=1
#SBATCH --ntasks-per-node=1
#SBATCH --cpus-per-task=4
#SBATCH --mem=8G
#SBATCH -t 00:30:00
#SBATCH --account={ACCOUNT}
#SBATCH --export=ALL

module reset
module load cpu/0.17.3b
module load gcc/10.2.0
source ~/.bashrc
source ~/miniconda3/etc/profile.d/conda.sh
conda activate rnaenv

python -u {SCRIPT_DIR}/process_single_gene_output.py "{gene_name}" "{chromosome}" "{start}" "{end}"
""")

    if DRY_RUN:
        print(f"[DRY RUN] [{gene_index}/{total}] Would submit {gene_name} "
              f"({chromosome}:{start}-{end})")
        print(f"           sbatch script: {sbatch_file}")
        return "DRY_RUN"

    result = subprocess.run(
        ["sbatch", "--parsable", sbatch_file],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        print(f"[output] ERROR submitting {gene_name}: {result.stderr.strip()}")
        return None

    job_id = result.stdout.strip()
    print(f"[output] [{gene_index}/{total}] Submitted {gene_name} "
          f"(Job {job_id}). Waiting...")

    while True:
        q = subprocess.run(
            ["squeue", "-h", "-j", job_id],
            capture_output=True, text=True,
        )
        if job_id not in q.stdout:
            print(f"[output] Job {job_id} ({gene_name}) finished.\n")
            break
        time.sleep(POLL_INTERVAL)

    return job_id


# ─────────────────────────────────────────────────────────────────────────────
# Main loop — every gene in the list has both pickles by construction
# ─────────────────────────────────────────────────────────────────────────────

submitted = 0
skipped   = 0
errors    = 0

for i, row in enumerate(all_genes.itertuples(index=False), 1):
    gene_name  = row.gene
    chromosome = str(row.chromosome)
    start      = int(row.start)
    end        = int(row.end)

    if already_done(gene_name):
        print(f"[output] [{i}/{total}] SKIPPING {gene_name} (output already exists)")
        skipped += 1
        continue

    result = submit_gene(gene_name, chromosome, start, end, i)

    if result is None:
        errors += 1
    else:
        submitted += 1

# ─────────────────────────────────────────────────────────────────────────────
# Final summary
# ─────────────────────────────────────────────────────────────────────────────

print(f"\n{'=' * 60}")
if DRY_RUN:
    print(f"  DRY RUN complete.")
    print(f"  Would submit: {submitted}")
    print(f"  Would skip (already done): {skipped}")
    print(f"  Set DRY_RUN = False and LIMIT_TEST = None to run for real.")
else:
    print(f"  Done.")
    print(f"  Submitted: {submitted}")
    print(f"  Skipped (already done): {skipped}")
    print(f"  Errors: {errors}")
print(f"{'=' * 60}")

In [ ]:
#!/bin/bash
# run_output_pipeline.sh
# Submits submit_output.py as a single SLURM orchestrator job.
# Run this after (or alongside) run_pipeline.sh — it will poll for
# extract pickles on its own and submit output jobs as they become ready.

source ~/miniconda3/etc/profile.d/conda.sh
conda activate rnaenv

SCRIPT_DIR="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)"
DATA_DIR="/expanse/lustre/projects/csd933/aho2/RNA_Mod/Data"
LOG_DIR="$DATA_DIR/full_hg38/pipeline_logs"
mkdir -p "$LOG_DIR"

# ── Write the orchestrator SLURM script ──────────────────────────────────────
cat > "$LOG_DIR/run_output.sh" << 'EOF'
#!/bin/bash
#SBATCH --job-name="pipeline_output"
#SBATCH --output="/expanse/lustre/projects/csd933/aho2/RNA_Mod/Data/full_hg38/pipeline_logs/output.log"
#SBATCH --error="/expanse/lustre/projects/csd933/aho2/RNA_Mod/Data/full_hg38/pipeline_logs/output.err"
#SBATCH --partition=shared
#SBATCH --nodes=1
#SBATCH --ntasks-per-node=1
#SBATCH --cpus-per-task=1
#SBATCH --mem=4G
#SBATCH -t 48:00:00
#SBATCH --account=csd922
#SBATCH --export=ALL

module reset
module load cpu/0.17.3b
module load gcc/10.2.0
source ~/.bashrc
source ~/miniconda3/etc/profile.d/conda.sh
conda activate rnaenv

python -u SCRIPT_DIR_PLACEHOLDER/submit_output.py
EOF

# Replace placeholder with actual script dir
sed -i "s|SCRIPT_DIR_PLACEHOLDER|$SCRIPT_DIR|g" "$LOG_DIR/run_output.sh"

# ── Submit ────────────────────────────────────────────────────────────────────
JOB_OUT=$(sbatch --parsable "$LOG_DIR/run_output.sh")

echo "============================================"
echo "  Output pipeline submitted as SLURM job"
echo "  Output orchestrator: Job $JOB_OUT"
echo ""
echo "  This job will survive notebook closure."
echo "  It polls for extract pickles automatically"
echo "  and submits one output job per gene as"
echo "  each gene's extracts become available."
echo ""
echo "  Track log:"
echo "    tail -f $LOG_DIR/output.log"
echo ""
echo "  Check status:"
echo "    squeue -u $USER"
echo "============================================"

### Webpage

Now that we have processed all of our genes of interest, I exported them into a webpage for visualization and ease of use.

Send files to database:

In [ ]:
"""
upload_to_supabase.py — run once on Expanse to populate Supabase.
Resumes automatically: skips genes already in the database.
"""

import pickle
import re
import os
from pathlib import Path
import pandas as pd
from supabase import create_client

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION — fill these in
# ─────────────────────────────────────────────────────────────────────────────

SUPABASE_URL = "https://htincjuvpxcztvmcslvn.supabase.co"   # ← your project URL
SUPABASE_KEY = "sb_publishable_uRFOq-HnZnJ4jjkpnWzvZg_52s-R6ri"       # ← your service_role key

DATABASE_DIR = Path("/expanse/lustre/projects/csd933/aho2/RNA_Mod/Data/full_hg38/test/processed_gene")

BATCH_SIZE = 100   # rows per upsert call

# ─────────────────────────────────────────────────────────────────────────────

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

def extract_mean(val):
    match = re.match(r"([0-9.]+)", str(val))
    return float(match.group(1)) if match else None

def clean_value(v):
    """Convert numpy/pandas types and NaN to JSON-safe Python types."""
    import numpy as np
    if isinstance(v, (np.integer,)):
        return int(v)
    if isinstance(v, (np.floating,)):
        return None if np.isnan(v) else float(v)
    if isinstance(v, float) and (v != v):   # NaN check
        return None
    try:
        if pd.isna(v):
            return None
    except (TypeError, ValueError):
        pass
    return v

# ── Get already-uploaded genes so we can skip them ───────────────────────────

print("Fetching already-uploaded genes...")
existing = set()
offset = 0
while True:
    result = (
        supabase.table("gene_modifications")
        .select("gene")
        .range(offset, offset + 999)
        .execute()
    )
    if not result.data:
        break
    for row in result.data:
        existing.add(row["gene"])
    if len(result.data) < 1000:
        break
    offset += 1000

print(f"  {len(existing)} genes already in database.")

# ── Upload loop ───────────────────────────────────────────────────────────────

pkl_files = sorted(DATABASE_DIR.glob("*.pkl"))
total     = len(pkl_files)
uploaded  = 0
skipped   = 0
errors    = 0

print(f"  {total} pickle files found.\n")

for i, pkl_file in enumerate(pkl_files, 1):
    gene = pkl_file.stem

    if gene in existing:
        print(f"[{i}/{total}] SKIP {gene} (already uploaded)")
        skipped += 1
        continue

    try:
        with open(pkl_file, "rb") as f:
            df = pickle.load(f)

        # Extract means from "0.829450 ± 0.236811" strings
        df["mr01_1_mean"] = df["MR01-1"].apply(extract_mean)
        df["mr01_2_mean"] = df["MR01-2"].apply(extract_mean)

        # Rename to DB-safe column names
        df = df.rename(columns={
            "Gene":         "gene",
            "Feature":      "feature",
            "Modification": "modification",
            "Count_MR01-1": "count_mr01_1",
            "CPK_MR01-1":   "cpk_mr01_1",
            "MR01-1":       "mr01_1",
            "Count_MR01-2": "count_mr01_2",
            "CPK_MR01-2":   "cpk_mr01_2",
            "MR01-2":       "mr01_2",
        })

        records = []
        for row in df.to_dict(orient="records"):
            records.append({k: clean_value(v) for k, v in row.items()})

        # Upload in batches
        for b in range(0, len(records), BATCH_SIZE):
            supabase.table("gene_modifications").upsert(
                records[b:b + BATCH_SIZE]
            ).execute()

        print(f"[{i}/{total}] Uploaded {gene} ({len(records)} rows)")
        uploaded += 1

    except Exception as e:
        print(f"[{i}/{total}] ERROR {gene}: {e}")
        errors += 1

print(f"\nDone. Uploaded: {uploaded}  Skipped: {skipped}  Errors: {errors}")

Code and repo are available here: https://github.com/Tofulati/rna_analysis

## Using the Webpage

> When first loading up the page, it will take a little time to load as it pulls data from the database. 

When navigating the webpage, you can select your sample (MR01-1 or MR01-2), region of interest (UTR 3', UTR 5', Intron, Exon), and Mod Type (A-to-I, m6A, or both) at the top of the page. Once selecting one of these values, the page will populate with a scatter graph that shows the various genes of interest that fall within your query. You can hover over the points and see the genes of interest with some details appearing as well. Scrolling past the scatter graph, you will find a table of genes which you can search through. Or, you can use the search bar provided to easily search for the gene that you are interested in. Once searching for the gene, you can select it, and a table will populate below. This table will contain the information about gene's modification probabilities, means, and CPKs for that particular sample you selected. If you want to switch to another sample, just change the sample at the top of the page, search the gene again, select, and then you can view the second sample's details.

*Sample View:*

POV: I want to view the modifications for AZIN1 in the MR01-1 sample.

1. Select MR01-1 in the top bar
2. Scroll down to search bar
3. Search up 'AZIN1'
4. Select 'AZIN1' from the table
5. Look at results